look out for the module versions 

In [1]:
import torch
import pandas
import numpy 
import sklearn
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding

print("✅ PyTorch:", torch.__version__)
print("✅ CUDA:", torch.cuda.is_available())
print("✅ Pandas:", pandas.__version__)
print("✅ NumPy:", numpy.__version__)
print("✅ Scikit-learn:", sklearn.__version__)
print("✅ torch:", torch.__version__)
print(f"Transformers version: {transformers.__version__}")

c:\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ PyTorch: 2.6.0+cu124
✅ CUDA: True
✅ Pandas: 3.0.1
✅ NumPy: 2.4.3
✅ Scikit-learn: 1.8.0
✅ torch: 2.6.0+cu124
Transformers version: 5.5.4


In [2]:
# 1. Vérifier si CUDA disponible
print(f"CUDA disponible: {torch.cuda.is_available()}")

# 2. Informations GPU
if torch.cuda.is_available():
    print(f"\n📊 INFORMATIONS GPU:")
    print(f"Nom: {torch.cuda.get_device_name(0)}")
    print(f"VRAM totale: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"VRAM utilisée: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
    print(f"VRAM réservée: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")
    print(f"Capacité: {torch.cuda.get_device_capability(0)}")
    print(f"Multi-processeurs: {torch.cuda.get_device_properties(0).multi_processor_count}")
    
    # 3. Version CUDA
    print(f"\n🔧 VERSIONS:")
    print(f"CUDA version (PyTorch): {torch.version.cuda}")
    print(f"cuDNN version: {torch.backends.cudnn.version()}")

else:
    print("❌ Aucun GPU détecté")

CUDA disponible: True

📊 INFORMATIONS GPU:
Nom: NVIDIA GeForce RTX 3080 Ti
VRAM totale: 12.88 GB
VRAM utilisée: 0.00 GB
VRAM réservée: 0.00 GB
Capacité: (8, 6)
Multi-processeurs: 80

🔧 VERSIONS:
CUDA version (PyTorch): 12.4
cuDNN version: 90100


test the XLM_R transformers 

In [15]:
# installer le modele en local 
# model_name = "xlm-roberta-base"
# tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir="./model")
# print("✅ tokenizer téléchargés dans ./model")
# model = AutoModelForSequenceClassification.from_pretrained( model_name, num_labels=5,cache_dir="./model")
# print("✅ Modèle téléchargés dans ./model")

In [3]:
# appeler le modele 
standardModelPath = "local_models\Roberta_model\models--xlm-roberta-base\snapshots\e73636d4f797dec63c3081bb6ed5c7b0bb3f2089"

tokenizer = AutoTokenizer.from_pretrained(standardModelPath)
model_lev1 = AutoModelForSequenceClassification.from_pretrained(standardModelPath,num_labels=5)
# forcer le mode single label
model_lev1.config.problem_type = "single_label_classification" 

# Déplacer sur GPU
device = torch.device("cuda")
model_lev1 = model_lev1.to(device)

<>:2: SyntaxWarning: invalid escape sequence '\R'
<>:2: SyntaxWarning: invalid escape sequence '\R'
C:\Users\Administrator\AppData\Local\Temp\ipykernel_20160\568541418.py:2: SyntaxWarning: invalid escape sequence '\R'
  standardModelPath = "local_models\Roberta_model\models--xlm-roberta-base\snapshots\e73636d4f797dec63c3081bb6ed5c7b0bb3f2089"
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 8081.59it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: local_models\Roberta_model\models--xlm-roberta-base\snapshots\e73636d4f797dec63c3081bb6ed5c7b0bb3f2089
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.out_proj.bias    | MIS

In [4]:
# importer le dataset et le preparer 
macro_dataset = pandas.read_csv('DATA\MergedComplaints.csv')
macro_dataset = macro_dataset[ ["Commentaire client"  , "Predicted class"] ]
macro_dataset['Predicted class'] = macro_dataset['Predicted class'].map( {'Problème / Réclamation':0 , 'Inapproprié':1 , 'Question/Assistance':2 , 'Positif':3 , 'Suggestion':4})

sub_dataset = pandas.read_csv('DATA\MergedComplaints.csv')
sub_dataset = sub_dataset[ ["Commentaire client"  , "Classe associé"] ]
# enlever les vides
sub_dataset = sub_dataset[sub_dataset["Classe associé"].notna()]
sub_dataset['Classe associé'] = sub_dataset['Classe associé'].map( {'Pannes et coupures générales':0 , 'Déploiement de la fibre':1 , 'Retards d’installation ou de mise en service':2 , 'Réclamation service':3 , 'Espace Client + My Idoom':4})

# renommer les target colones en label , demander par le Trainer 
macro_dataset.rename(columns={"Predicted class":"label"} , inplace=True)
sub_dataset.rename(columns={"Classe associé":"label"} , inplace=True)

# enlever les nans , demander par trainer
sub_dataset = sub_dataset.dropna(subset=["label"])
macro_dataset = macro_dataset.dropna(subset=["label"])

# convertir les labels en type int ,demander par le trainer
macro_dataset['label'] = macro_dataset['label'].astype(int)
sub_dataset['label'] = sub_dataset['label'].astype(int)

<>:2: SyntaxWarning: invalid escape sequence '\M'
<>:6: SyntaxWarning: invalid escape sequence '\M'
<>:2: SyntaxWarning: invalid escape sequence '\M'
<>:6: SyntaxWarning: invalid escape sequence '\M'
C:\Users\Administrator\AppData\Local\Temp\ipykernel_20160\1719540064.py:2: SyntaxWarning: invalid escape sequence '\M'
  macro_dataset = pandas.read_csv('DATA\MergedComplaints.csv')
C:\Users\Administrator\AppData\Local\Temp\ipykernel_20160\1719540064.py:6: SyntaxWarning: invalid escape sequence '\M'
  sub_dataset = pandas.read_csv('DATA\MergedComplaints.csv')


In [5]:
print(macro_dataset["label"].value_counts())

label
0    18388
1     4426
2     3316
3     3072
4      769
Name: count, dtype: int64


In [6]:
# preparer le metric et data et tokenezied

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix , f1_score

# Split
df_macro_train , df_macro_test = train_test_split(macro_dataset , test_size=0.2,random_state=42)
df_sub_train , df_sub_test = train_test_split(sub_dataset , test_size=0.2,random_state=42)

# Datasets : rend les données structurée pour le modèle
train_macro_dataset = Dataset.from_pandas(df_macro_train[ ["Commentaire client","label"] ])
test_macro_dataset = Dataset.from_pandas(df_macro_test[ ["Commentaire client","label"] ])

train_sub_dataset = Dataset.from_pandas(df_sub_train[ ["Commentaire client","label"] ])
test_sub_dataset = Dataset.from_pandas(df_sub_test[ ["Commentaire client","label"] ])

# fonction de tokenizaion pour les 4 datasets 
def tokenization(batch):
    tokens = tokenizer(
        batch["Commentaire client"],
        truncation=True,
        padding="max_length",
        max_length=128
    )
    tokens["labels"] = [int(x) for x in batch["label"]]
    return tokens

# transfrom the complaints into tokens
tokenized_macro_train = train_macro_dataset.map(tokenization,batched=True)
tokenized_macro_test = test_macro_dataset.map(tokenization,batched=True)
tokenized_sub_train = train_sub_dataset.map(tokenization,batched=True)
tokenized_sub_test = test_sub_dataset.map(tokenization,batched=True)

# delet complaints 
tokenized_macro_train = tokenized_macro_train.remove_columns(["Commentaire client"])
tokenized_macro_test = tokenized_macro_test.remove_columns(["Commentaire client"])
tokenized_sub_train = tokenized_sub_train.remove_columns(["Commentaire client"])
tokenized_sub_test = tokenized_sub_test.remove_columns(["Commentaire client"])


Map: 100%|██████████| 3678/3678 [00:00<00:00, 34564.18 examples/s]


In [7]:
# Métriques
def computeMetrics(eval_pred):
    logits, labels = eval_pred
    predictions = numpy.argmax(logits, axis=-1)

    # métriques par classe
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average=None
    )

    # accuracy globale
    acc = accuracy_score(labels, predictions)

    # matrice de confusion
    cm = confusion_matrix(labels, predictions)

    f1_macro = f1_score(labels, predictions, average="macro")

    # construire dictionnaire
    metrics = {
        "accuracy": acc
    }
    metrics["f1"] = f1_macro

    # ajouter métriques par classe
    for i in range(len(precision)):
        metrics[f'precision_class_{i}'] = precision[i]
        metrics[f'recall_class_{i}'] = recall[i]
        metrics[f'f1_class_{i}'] = f1[i]
    
    # revoir ca 
    print("\n📊 Confusion Matrix:\n", cm)

    return metrics

In [8]:
# Après avoir créé le dataset, forcer le type des labels
tokenized_macro_train = tokenized_macro_train.map(lambda x: {"labels": torch.tensor(x["labels"], dtype=torch.long)})
tokenized_macro_test = tokenized_macro_test.map(lambda x: {"labels": torch.tensor(x["labels"], dtype=torch.long)})
tokenized_sub_train = tokenized_sub_train.map(lambda x: {"labels": torch.tensor(x["labels"], dtype=torch.long)})
tokenized_sub_test = tokenized_sub_test.map(lambda x: {"labels": torch.tensor(x["labels"], dtype=torch.long)})

# rend les séquences de même taille dans chaque batch automatiquement
dataCollator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/23976 [00:00<?, ? examples/s]

Map: 100%|██████████| 3678/3678 [00:00<00:00, 33793.39 examples/s]


In [9]:
result_path = './RESULT/'

training_args = TrainingArguments(

    output_dir= result_path + "roberta_finetuned_lev1",
    learning_rate=2e-5,  #standard 
    per_device_train_batch_size=16,  
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1, # augmente ? 

    num_train_epochs=5, # repasse
    weight_decay=0.01, # to avoid overfit in words !
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True, # save the best of epoch

    fp16=True, # 16 bit precision 

    dataloader_num_workers= 8 , # rule = num_CPU_cores / num_GPUs
    logging_steps=100,         

    metric_for_best_model="f1",
    greater_is_better=True
)

trainer_lev1 = Trainer(
    model=model_lev1,
    args=training_args,
    train_dataset=tokenized_macro_train,
    eval_dataset=tokenized_macro_test,
    data_collator=dataCollator,
    compute_metrics=computeMetrics
)

In [10]:
# entrainer le modele 
trainer_lev1.train()

# Sauvegarder le modele 
model_lev1.save_pretrained(result_path +'model_classifier_lv1')
tokenizer.save_pretrained(result_path +'tokenizer1')
print("Modèle niveau 1 sauvegardé")

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision Class 0,Recall Class 0,F1 Class 0,Precision Class 1,Recall Class 1,F1 Class 1,Precision Class 2,Recall Class 2,F1 Class 2,Precision Class 3,Recall Class 3,F1 Class 3,Precision Class 4,Recall Class 4,F1 Class 4
1,0.308380,0.304048,0.914429,0.852247,0.927326,0.961370,0.944041,0.864242,0.804740,0.833431,0.879234,0.879234,0.879234,0.964824,0.917197,0.940408,0.790909,0.572368,0.664122
2,0.185459,0.254987,0.939283,0.886467,0.945903,0.977260,0.961326,0.929306,0.816027,0.868990,0.911268,0.952872,0.931605,0.958533,0.957006,0.957769,0.853211,0.611842,0.712644
3,0.148884,0.224181,0.961134,0.924801,0.964171,0.987945,0.975913,0.951220,0.880361,0.914420,0.977444,0.957290,0.967262,0.974318,0.966561,0.970424,0.809524,0.782895,0.795987
4,0.062228,0.196392,0.969475,0.938423,0.972013,0.989589,0.980722,0.958726,0.917607,0.937716,0.980568,0.966127,0.973294,0.979200,0.974522,0.976856,0.868613,0.782895,0.823529
5,0.058184,0.189926,0.972310,0.942624,0.975669,0.988767,0.982174,0.961583,0.932280,0.946705,0.976366,0.973490,0.974926,0.982400,0.977707,0.980048,0.881481,0.782895,0.829268



📊 Confusion Matrix:
 [[3509   64   57    3   17]
 [ 140  713   13   14    6]
 [  64   18  597    0    0]
 [  25   22    5  576    0]
 [  46    8    7    4   87]]


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.08s/it]



📊 Confusion Matrix:
 [[3567   32   39    4    8]
 [ 125  723   17   17    4]
 [  24    6  647    0    2]
 [  13   12    0  601    2]
 [  42    5    7    5   93]]


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.12s/it]


...


📊 Confusion Matrix:
 [[3612   19    9    2    8]
 [  57  813    3   10    3]
 [  16    5  656    0    2]
 [   5    6    0  612    5]
 [  26    5    1    1  119]]


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.04s/it]



📊 Confusion Matrix:
 [[3609   20   10    2    9]
 [  48  826    3    7    2]
 [  14    2  661    0    2]
 [   5    5    1  614    3]
 [  23    6    2    2  119]]


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.03s/it]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.la

Modèle niveau 1 sauvegardé


In [ ]:
# petit test model_classifier_lv1
model_lev1 = AutoModelForSequenceClassification.from_pretrained('RESULT\model_classifier_lv1')
tokenizer1 = AutoTokenizer.from_pretrained('RESULT\Tokenizer1')

def predict(text):
    inputs = tokenizer1(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = model_lev1(**inputs)

    logits = outputs.logits
    pred = torch.argmax(logits, dim=1).item()

    id2label = {0:'Problème / Réclamation' , 1:'Inapproprié' , 2:'Question/Assistance' , 3:'Positif' , 4:'Suggestion'}

    return id2label[pred]

In [ ]:
text = "لي معندوش أهلية الخط واش يدير"

print([predict(text)])

next model level 2
qui classifie si cest une reclamation 

In [9]:
model_lev2 = AutoModelForSequenceClassification.from_pretrained(standardModelPath,num_labels=5)

# Déplacer sur GPU
model_lev2 = model_lev2.to(device)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 14439.86it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: local_models\Roberta_model\models--xlm-roberta-base\snapshots\e73636d4f797dec63c3081bb6ed5c7b0bb3f2089
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

In [15]:
from collections import Counter

Counter(tokenized_sub_train["label"])
# prendre que les valeurs et les mettres dans une liste 
# counts_list = list(Counter(tokenized_sub_train["label"]).values())
# print(counts_list)

Counter({0: 2054, 1: 1837, 2: 1385, 3: 1146, 4: 933})

In [16]:
# probleme : class 3 est confendu par 0 et 4 alors je utilise class weight 
# solution : utiliser class weight --> cree un Trainer personalise 
 
# first i need les weights of each label in the train data 
class_counts = numpy.array([2054, 1837, 1385, 1460, 933])

N_total = class_counts.sum()
num_classes = len(class_counts)
class_weights = N_total / (num_classes * class_counts)

# convertire en tenseur et donner au GPU
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

In [17]:
# code depuis chatgpt : une class qui herite de Trainer (TRes interesant )
class WeightedLossTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
    
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get('logits')
        
        # Calculer la loss avec les poids
        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), 
                        labels.view(-1))
        
        return (loss, outputs) if return_outputs else loss

In [18]:

training_args = TrainingArguments(

    output_dir= result_path + "roberta_finetuned_lev2",
    learning_rate=2e-5,  
    per_device_train_batch_size=24,  
    per_device_eval_batch_size=24,
    gradient_accumulation_steps=1,  

    num_train_epochs=5, 
    weight_decay=0.01, 
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True, 

    fp16=True, 

    dataloader_num_workers= 8 , 
    logging_steps=100,         
    
    metric_for_best_model="f1",
    greater_is_better=True
)

trainer_lev2 = WeightedLossTrainer( 
    model=model_lev2,
    args=training_args,
    train_dataset=tokenized_sub_train,
    eval_dataset=tokenized_sub_test,
    data_collator=dataCollator,
    compute_metrics=computeMetrics , 
    class_weights = class_weights_tensor
)

In [19]:
# entrainer le modele 
trainer_lev2.train()

# Sauvegarder le modele 
model_lev2.save_pretrained(result_path +'model_classifier_lv2')
tokenizer.save_pretrained(result_path +'tokenizer2')
print("Modèle niveau 2 sauvegardé")

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision Class 0,Recall Class 0,F1 Class 0,Precision Class 1,Recall Class 1,F1 Class 1,Precision Class 2,Recall Class 2,F1 Class 2,Precision Class 3,Recall Class 3,F1 Class 3,Precision Class 4,Recall Class 4,F1 Class 4
1,0.707604,0.620834,0.779772,0.771243,0.787755,0.773547,0.780586,0.863341,0.854077,0.858684,0.724638,0.810811,0.765306,0.628821,0.498270,0.555985,0.840816,0.958140,0.895652
2,0.491737,0.511036,0.817292,0.813369,0.818540,0.831663,0.825050,0.872807,0.854077,0.863341,0.770574,0.835135,0.801556,0.726891,0.598616,0.656546,0.877637,0.967442,0.920354
3,0.393028,0.497826,0.840674,0.836954,0.856000,0.857715,0.856857,0.909297,0.860515,0.884234,0.789082,0.859459,0.822768,0.734848,0.671280,0.701627,0.887446,0.953488,0.919283
4,0.327248,0.479882,0.845568,0.841290,0.858847,0.865731,0.862275,0.888172,0.886266,0.887218,0.810390,0.843243,0.826490,0.748031,0.657439,0.699816,0.896552,0.967442,0.930649



📊 Confusion Matrix:
 [[386  14  27  50  22]
 [ 12 398  40  12   4]
 [ 13  37 300  17   3]
 [ 76  12  47 144  10]
 [  3   0   0   6 206]]


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.09s/it]



📊 Confusion Matrix:
 [[415  15  18  36  15]
 [ 14 398  38  13   3]
 [ 15  31 309  13   2]
 [ 59  12  36 173   9]
 [  4   0   0   3 208]]


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.12s/it]



📊 Confusion Matrix:
 [[428   8   8  41  14]
 [ 12 401  39  12   2]
 [ 16  24 318  10   2]
 [ 42   8  37 194   8]
 [  2   0   1   7 205]]


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.22s/it]



📊 Confusion Matrix:
 [[432  11   7  37  12]
 [  9 413  32  10   2]
 [ 13  30 312  13   2]
 [ 46  11  34 190   8]
 [  3   0   0   4 208]]


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.17s/it]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.la

Modèle niveau 2 sauvegardé


In [ ]:
# petit test model_classifier_lv2
model_lev2 = AutoModelForSequenceClassification.from_pretrained('RESULT\model_classifier_lv2')
tokenizer2 = AutoTokenizer.from_pretrained('RESULT\Tokenizer2')

def predict(text):
    inputs = tokenizer2(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = model_lev2(**inputs)

    logits = outputs.logits
    pred = torch.argmax(logits, dim=1).item()

    id2label = {0:'Pannes et coupures générales' , 1:'Déploiement de la fibre' , 2:'Retards d’installation ou de mise en service' , 3:'Réclamation service' , 4:'Espace Client + My Idoom'}

    return id2label[pred]

In [ ]:
test_text = 'فضاء الزبون مغلوو و قالموقع الإلكتروني معطل'

print(f"the predicted class is : {[predict(test_text)]}")

SENTIMENT ANALYSIS

In [14]:
model_sentiment = AutoModelForSequenceClassification.from_pretrained(standardModelPath,num_labels=3)
# mode single label
model_sentiment.config.problem_type = "single_label_classification" 

# Déplacer sur GPU
model_sentiment = model_sentiment.to(device)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 14229.73it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: local_Roberta_model\models--xlm-roberta-base\snapshots\e73636d4f797dec63c3081bb6ed5c7b0bb3f2089
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Con

In [27]:
# (les donnes contient deja sarcasm )

# preparer le data 
sentiment_dataset = pandas.read_csv('DATA\MergedComplaints.csv')
sentiment_dataset = sentiment_dataset[ ["Commentaire client"  , "Predicted class"] ]
sentiment_dataset["Predicted class"] = (
    sentiment_dataset["Predicted class"]
    .str.strip()
    .str.lower()
)
sentiment_dataset = sentiment_dataset.dropna(subset=["Predicted class"])

mapping = {
    'problème / réclamation': 0,
    'inapproprié': 0,
    'question/assistance': 1,
    'positif': 2,
    'suggestion': 1
}
sentiment_dataset["label"] = sentiment_dataset["Predicted class"].map(mapping)

sentiment_dataset = sentiment_dataset.drop(columns=['Predicted class'])

# renommer les target colones en label , demander par le Trainer 
sentiment_dataset.rename(columns={"sentiment class":"label"} , inplace=True)

# probleme : la class 0 et trop grande compare au autre = overfit 
# solution : reduire sa taille et augmenter les autres 

from sklearn.utils import resample

df = sentiment_dataset.copy()
df_majority = df[df["label"] == 0]
df_rest = df[df["label"] != 0]
df_majority_reduced = resample(
    df_majority,
    replace=False,
    n_samples=int(len(df_majority) * 0.6),
    random_state=42
)
sentiment_dataset = pandas.concat([
    df_majority_reduced,
    df_rest
])

# convertir les labels en type int ,demander par le trainer
sentiment_dataset['label'] = sentiment_dataset['label'].astype(int)

<>:4: SyntaxWarning: invalid escape sequence '\M'
<>:4: SyntaxWarning: invalid escape sequence '\M'
C:\Users\Administrator\AppData\Local\Temp\ipykernel_21068\3457890901.py:4: SyntaxWarning: invalid escape sequence '\M'
  sentiment_dataset = pandas.read_csv('DATA\MergedComplaints.csv')


In [28]:
# preparer le metric et data et tokenezied

# Split
df_sentiment_train , df_sentiment_test = train_test_split(sentiment_dataset , test_size=0.2,random_state=42)

# Datasets
train_sentiment_dataset = Dataset.from_pandas(df_sentiment_train[ ["Commentaire client","label"] ])
test_sentiment_dataset = Dataset.from_pandas(df_sentiment_test[ ["Commentaire client","label"] ])

# fonction de tokenizaion pour les 4 datasets 
def tokenization(batch):
    tokens = tokenizer(
        batch["Commentaire client"],
        truncation=True,
        padding="max_length",
        max_length=128
    )
    tokens["labels"] = [int(x) for x in batch["label"]]
    return tokens

# transfrom the complaints into tokens
tokenized_sentiment_train = train_sentiment_dataset.map(tokenization,batched=True)
tokenized_sentiment_test = test_sentiment_dataset.map(tokenization,batched=True)


# delete texts to consunme memory 
tokenized_sentiment_train = tokenized_sentiment_train.remove_columns(["Commentaire client"])
tokenized_sentiment_test = tokenized_sentiment_test.remove_columns(["Commentaire client"])

# metric existe deja 

Map: 100%|██████████| 2156/2156 [00:00<00:00, 32809.61 examples/s]


In [29]:
from collections import Counter

Counter(tokenized_sentiment_train["label"])

Counter({0: 5427, 1: 1940, 2: 1255})

In [33]:
# probleme : positif na pas de poid 
# solution : reduire la taille de negatif + donner plus de poid au negatif et neutre  --> cree un Trainer personalise 
 
# first i need les weights of each label in the train data sentiment
class_counts = numpy.array([5427,1957,1215])

N_total = class_counts.sum()
class_weights = N_total / ( 3 * class_counts )

# convertire en tenseur et donner au GPU
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

In [34]:
training_args = TrainingArguments(

    output_dir= result_path + "roberta_finetuned_sentiment",
    learning_rate=2e-5,  
    per_device_train_batch_size=24,  
    per_device_eval_batch_size=24,
    gradient_accumulation_steps=1,  

    num_train_epochs=3, 
    weight_decay=0.01, 
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True, 

    fp16=True, 

    dataloader_num_workers= 8 , 
    logging_steps=100,         
    greater_is_better=False,
)

# apres tester avec ca
trainer_sentiment = WeightedLossTrainer( 
    model=model_sentiment,
    args=training_args,
    train_dataset=tokenized_sentiment_train,
    eval_dataset=tokenized_sentiment_test,
    data_collator=dataCollator,
    compute_metrics=computeMetrics , 
    class_weights = class_weights_tensor
)

In [35]:
# entrainer le modele 
trainer_sentiment.train()

# Sauvegarder le modele 
model_sentiment.save_pretrained(result_path +'model_sentimentAnalysis')
tokenizer.save_pretrained(result_path +'tokenizer_sentiment')
print("Modèle analyse de sentiment sauvegardé")

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision Class 0,Recall Class 0,F1 Class 0,Precision Class 1,Recall Class 1,F1 Class 1,Precision Class 2,Recall Class 2,F1 Class 2
1,0.170969,0.392591,0.933673,0.921347,0.949676,0.951729,0.950702,0.913607,0.868583,0.890526,0.890728,0.957295,0.922813
2,0.105654,0.398232,0.938312,0.927728,0.948645,0.958213,0.953405,0.927632,0.868583,0.897137,0.906040,0.960854,0.932642
3,0.056348,0.376298,0.934601,0.924512,0.958884,0.940922,0.949818,0.888889,0.903491,0.896130,0.899666,0.957295,0.927586



📊 Confusion Matrix:
 [[1321   39   28]
 [  59  423    5]
 [  11    1  269]]


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.52s/it]



📊 Confusion Matrix:
 [[1330   33   25]
 [  61  423    3]
 [  11    0  270]]


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.51s/it]



📊 Confusion Matrix:
 [[1306   55   27]
 [  44  440    3]
 [  12    0  269]]


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.46s/it]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.la

Modèle analyse de sentiment sauvegardé


In [36]:
model_sentiment = AutoModelForSequenceClassification.from_pretrained('RESULT/model_sentimentAnalysis')
tokenizer = AutoTokenizer.from_pretrained('RESULT/model_sentimentAnalysis')

model_sentiment.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_sentiment.to(device)

def predict(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model_sentiment(**inputs)

    logits = outputs.logits
    pred = torch.argmax(logits, dim=1).item()

    id2label = {0:'negative', 1:'neutral', 2:'positif'}

    return id2label[pred]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 10281.79it/s]


In [ ]:
test_text = ''
print([predict(test_text)])

ALTERNATIVE

In [1]:
from transformers import BertTokenizer, BertForSequenceClassification

model_name = "alger-ia/dziribert_sentiment"

tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name)

tokenizer.save_pretrained("local_models/dziribert_local")
model.save_pretrained("local_models/dziribert_local")

print("Modèle enregistré en local")

c:\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 12460.91it/s]
BertForSequenceClassification LOAD REPORT from: alger-ia/dziribert_sentiment
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.11it/s]

Modèle enregistré en local


In [9]:
import torch.nn.functional as F
from transformers import BertTokenizer, BertForSequenceClassification

local_path = "local_models/dziribert_local"
tokenizer_dziri_sentiment = BertTokenizer.from_pretrained(local_path)
model_dziri_sentiment = BertForSequenceClassification.from_pretrained(local_path)

device = torch.device("cuda")
model_dziri_sentiment = model_dziri_sentiment.to(device)

texte_algerien = "c'est pas possible faite vite" 
inputs = tokenizer(texte_algerien, return_tensors="pt", truncation=True, padding=True)

with torch.no_grad(): # On désactive le calcul des gradients pour l'inférence
    outputs = model(**inputs)

logits = outputs.logits
probabilities = F.softmax(logits, dim=-1)

predicted_class = torch.argmax(probabilities, dim=-1).item()

labels = {
    0: "négatif",
    1: "neutre",
    2: "positif"
}

print(f"Classe prédite : {labels[predicted_class]}")
print(f"Probabilités : {probabilities}")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 15764.23it/s]


Classe prédite : négatif
Probabilités : tensor([[0.9672, 0.0036, 0.0292]])


GESTION DE PRIORITE